In [29]:
import pandas as pd

sheets = pd.read_excel("indicatorsMY_v02.xlsx", sheet_name=None)

# Vérifie les noms des feuilles
print(sheets.keys())

# Exemple : afficher les premières lignes de la feuille 'Alternatives'
df = sheets['Alternatives']
df.shape

dict_keys(['Alternatives', 'Market_Indicators', 'Price_Level_BTC', 'Open_Interests_Futures', 'Money_Supply', 'Central_Banks_Balance_Sheets', 'Central_Banks_Policy_Rates', 'TED_Spreads', '3M_Gov__Bonds', 'Yield_Curve', 'Foreign_Exchange', 'Miscellaneous', 'Gross_Domestic_Products', 'Industrial_Production', 'Retail_Sales', 'Inflation', 'Economic Policy Uncertainty', 'News Based Policy Uncertainty', 'Economic_Surprises', 'Semiconductor_Market_Dynamics', 'Sentiment', 'OnChain'])


(136, 5869)

In [30]:
import pandas as pd

ALLOWED_VARS = {"px_open","px_last","px_high","px_low","px_volume"}
DEFAULT_VAR = "px_last"  # variable par défaut pour les séries type Sentiment/OnChain

df_list = []

for name, sheet in sheets.items():
    sheet = sheet.dropna(how="all", axis=0).dropna(how="all", axis=1)
    if sheet.shape[1] < 3:
        continue

    a1 = str(sheet.columns[0]).strip().lower()
    b1 = str(sheet.columns[1]).strip().upper()

    # =========================
    # FORMAT PRIX (Alternatives)
    # A=ticker, B=px_last..., C..=dates ; B1="DATES"
    # =========================
    if b1 == "DATES":
        tmp = sheet.copy()
        tmp.columns = ["Ticker", "Variable"] + list(tmp.columns[2:])

        # garder uniquement les 5 variables prix
        tmp["Variable"] = tmp["Variable"].astype(str).str.strip()
        tmp = tmp[tmp["Variable"].isin(ALLOWED_VARS)]

        melted = tmp.melt(
            id_vars=["Ticker", "Variable"],
            var_name="Date",
            value_name="Value"
        )

    # =========================
    # FORMAT FEATURES (Sentiment/OnChain)
    # A="date" puis ligne 2 = feature, B..=dates
    # Ici : on veut feature => Ticker et Variable = px_last (fixe)
    # =========================
    elif a1 == "date":
        tmp = sheet.copy()

        # Col A contient les "tickers" (fear_and_greed, TxCnt, etc.)
        tmp = tmp.rename(columns={tmp.columns[0]: "Ticker"})
        tmp.insert(1, "Variable", DEFAULT_VAR)

        melted = tmp.melt(
            id_vars=["Ticker", "Variable"],
            var_name="Date",
            value_name="Value"
        )

    else:
        continue

    # Nettoyage Date + Value
    melted["Date"] = pd.to_datetime(melted["Date"], errors="coerce")
    melted["Value"] = pd.to_numeric(
        melted["Value"].astype(str)
            .str.replace(",", ".", regex=False)
            .str.replace(" ", "", regex=False),
        errors="coerce"
    )
    melted = melted.dropna(subset=["Date", "Value"])

    melted["Ticker"] = melted["Ticker"].astype(str).str.strip()
    melted["Variable"] = melted["Variable"].astype(str).str.strip()

    melted["Category"] = name
    df_list.append(melted)

merged_df = pd.concat(df_list, ignore_index=True)

print("Variables disponibles :", sorted(merged_df["Variable"].unique()))
print("Tickers disponibles :", len(merged_df["Ticker"].unique()))


Variables disponibles : ['px_high', 'px_last', 'px_low', 'px_open', 'px_volume']
Tickers disponibles : 200


In [31]:
merged_df[['Category', 'Ticker', 'Variable']].drop_duplicates().head(50)

,Category,Ticker,Variable
0,Alternatives,DXY Curncy,px_open
1,Alternatives,DXY Curncy,px_last
2,Alternatives,DXY Curncy,px_high
3,Alternatives,DXY Curncy,px_low
4,Alternatives,XAU Curncy,px_open
5,Alternatives,XAU Curncy,px_last
6,Alternatives,XAU Curncy,px_high
7,Alternatives,XAU Curncy,px_low
8,Alternatives,SPX Index,px_open
9,Alternatives,SPX Index,px_last


In [32]:
merged_df["Value"].dtype, merged_df["Value"].head()

(dtype('float64'),
 0      76.34801
 1      76.12601
 2      76.66801
 3      76.02100
 4    1049.20000
 Name: Value, dtype: float64)

In [33]:
# Nettoyage global de la colonne Value
merged_df["Value"] = (
    merged_df["Value"]
    .astype(str)         # convertir en string
    .str.replace(",", ".")  # remplacer virgules
    .str.replace(" ", "")   # enlever espaces
    .str.replace("-", "")    # remplacer les tirets
)

# Conversion en float
merged_df["Value"] = pd.to_numeric(merged_df["Value"], errors="coerce")

In [34]:
# Convert all dates to datetime
merged_df["Date"] = pd.to_datetime(merged_df["Date"], errors="coerce")

# Force all dates to be timezone-naive
merged_df["Date"] = merged_df["Date"].dt.tz_localize(None)

In [35]:
pivot_df = merged_df.pivot_table(
    index="Date",
    columns="Ticker",
    values="Value",
    aggfunc='mean'   # corrige les doublons mélangés
)

corr = pivot_df.corr()
corr


Ticker,AdrActCnt,BASPTDSP Index,BDIY Index,BOJDPBAL Index,BlkCnt,CCMP Index,CESICNY Index,CESIEUR Index,CESIJPY Index,CESIUSD Index,...,mempool_growth,mempool_size_bytes,miner_revenue_usd,mining_difficulty,n_tx_excl_popular,price_usd,tx_fees_usd,tx_per_block,tx_per_second,utxo_count
Ticker,,,,,,,,,,,,,,,,,,,,,
AdrActCnt,1.000000,0.064268,0.485536,0.012550,0.060275,0.429490,0.054623,0.410440,0.146027,0.296876,...,0.284283,0.281877,0.571241,0.249534,0.392391,-0.093066,0.233444,0.391351,0.274072,0.357221
BASPTDSP Index,0.064268,1.000000,-0.272597,0.017739,-0.021686,-0.101936,0.031180,0.165752,-0.211383,0.021041,...,0.073628,-0.301102,-0.117918,-0.100492,0.176128,-0.074607,-0.132178,0.203814,0.104114,0.174945
BDIY Index,0.485536,-0.272597,1.000000,0.036976,-0.064570,0.256508,-0.069542,0.080558,0.180371,-0.050104,...,0.012356,0.069580,0.631554,0.258869,0.222922,-0.038660,0.163903,0.221326,0.132557,0.328234
BOJDPBAL Index,0.012550,0.017739,0.036976,1.000000,-0.033794,0.504691,-0.117894,-0.148542,-0.136384,-0.169340,...,0.052043,-0.064148,0.394868,0.763197,0.352002,-0.012156,-0.051594,0.344349,0.165110,0.609678
BlkCnt,0.060275,-0.021686,-0.064570,-0.033794,1.000000,-0.106879,-0.101834,-0.059860,-0.097325,-0.000118,...,0.040531,-0.027556,0.008288,-0.083498,-0.004266,-0.011690,0.005440,-0.111446,-0.001478,-0.073917
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
price_usd,-0.093066,-0.074607,-0.038660,-0.012156,-0.011690,-0.029718,-0.039382,-0.044256,-0.015390,-0.008565,...,-0.025197,-0.010471,-0.047317,-0.033123,-0.065255,1.000000,-0.014509,-0.070128,-0.031075,-0.056166
tx_fees_usd,0.233444,-0.132178,0.163903,-0.051594,0.005440,0.118711,-0.078650,0.139658,0.095130,0.060101,...,0.133237,0.378540,0.411672,0.125381,0.263347,-0.014509,1.000000,0.273559,0.175180,0.194710
tx_per_block,0.391351,0.203814,0.221326,0.344349,-0.111446,0.459758,-0.101433,0.027053,-0.145527,0.007722,...,0.299220,0.471387,0.498025,0.695349,0.877768,-0.070128,0.273559,1.000000,0.510983,0.788411


In [36]:
# Voir toutes les catégories disponibles
print("Catégories disponibles :")
print(merged_df['Category'].unique())
print("\nNombre total de catégories :", merged_df['Category'].nunique())

# Voir tous les tickers disponibles
print("\nTickers disponibles :")
print(merged_df['Ticker'].unique())
print("\nNombre total de tickers :", merged_df['Ticker'].nunique())

# Voir toutes les variables disponibles (px_open, px_last, etc.)
print("\nVariables disponibles :")
print(merged_df['Variable'].unique())
print("\nNombre total de variables :", merged_df['Variable'].nunique())

Catégories disponibles :
['Alternatives' 'Market_Indicators' 'Price_Level_BTC'
 'Open_Interests_Futures' 'Money_Supply' 'Central_Banks_Balance_Sheets'
 'Central_Banks_Policy_Rates' 'TED_Spreads' '3M_Gov__Bonds' 'Yield_Curve'
 'Foreign_Exchange' 'Miscellaneous' 'Gross_Domestic_Products'
 'Industrial_Production' 'Retail_Sales' 'Inflation'
 'Economic Policy Uncertainty' 'News Based Policy Uncertainty'
 'Economic_Surprises' 'Semiconductor_Market_Dynamics' 'Sentiment'
 'OnChain']

Nombre total de catégories : 22

Tickers disponibles :
['DXY Curncy' 'XAU Curncy' 'SPX Index' 'CCMP Index' 'RTY Index'
 'CRY Index' 'SPGSCITR Index' 'SXXP Index' 'SHCOMP Index' 'LUATTRUU Index'
 'LUACTRUU Index' 'LF98TRUU Index' 'NKY Index' 'XETUSD Curncy'
 'XLCUSD Curncy' 'XBNUSD Curncy' 'XRPUSD Curncy' 'XDGUSD Curncy'
 'XADUSD Curncy' 'XSOUSD Curncy' 'VIX Index' 'MOVE Index' 'CVIX Index'
 'RIOT US Equity' 'JPMVXYGL Index' 'MARA US Equity' 'CIFR US Equity'
 'IREN US Equity' 'CORZ US Equity' 'XBTUSD Curncy' 'CFC5Q

In [37]:
print(merged_df['Ticker'].nunique())

200


In [38]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from collections import defaultdict

# ============================================================
# CONFIGURATION
# ============================================================

START_DATE = "2018-03-01"
HORIZONS = [1, 14, 30, 60]
RETURN_TYPE = "simple"  # "simple" ou "log"

# ============================================================
# FONCTION POUR CALCULER LES RENDEMENTS (TARGET)
# ============================================================

def calculate_returns(prices, horizon, return_type="simple"):
    """
    Calcule les rendements futurs SANS data leakage

    return_type:
    - "simple": (P_t+h - P_t) / P_t
    - "log": ln(P_t+h / P_t) (via somme de log returns sur l'horizon)
    """
    prices = prices.astype(float)

    if return_type == "simple":
        future_price = prices.shift(-horizon)
        returns = (future_price - prices) / prices
    else:  # log
        log_prices = np.log(prices)
        log_ret = log_prices.diff()
        # somme des log-returns futurs sur horizon
        returns = log_ret.shift(-horizon).rolling(horizon).sum()

    return returns

# ============================================================
# FONCTION POUR CLASSIFIER PAR SCÉNARIOS
# ============================================================

def classify_scenario(returns):
    """
    Classifie les rendements en 5 scénarios basés sur l'écart-type (mean/std)

    Scénarios:
    - 0: Fortement baissier (< mean - 1 std)
    - 1: Baissier (mean -1 std à mean -0.5 std)
    - 2: Neutre (mean -0.5 std à mean +0.5 std)
    - 3: Haussier (mean +0.5 std à mean +1 std)
    - 4: Fortement haussier (> mean + 1 std)
    """
    mean_ret = returns.mean()
    std_ret = returns.std()

    scenarios = pd.Series(index=returns.index, dtype="Int64")

    scenarios[returns < mean_ret - std_ret] = 0
    scenarios[(returns >= mean_ret - std_ret) & (returns < mean_ret - 0.5 * std_ret)] = 1
    scenarios[(returns >= mean_ret - 0.5 * std_ret) & (returns <= mean_ret + 0.5 * std_ret)] = 2
    scenarios[(returns > mean_ret + 0.5 * std_ret) & (returns <= mean_ret + std_ret)] = 3
    scenarios[returns > mean_ret + std_ret] = 4

    return scenarios

# ============================================================
# FONCTION D'ÉVALUATION (CLASSIFICATION UNIQUEMENT)
# ============================================================

def evaluate_horizon_classification_only(df_original, horizon, return_type="simple"):
    """
    Évaluation pour un horizon donné (CLASSIFICATION UNIQUEMENT)
    Retourne accuracy + confusion matrix + top features + distrib scénarios
    """

    df = df_original.copy()
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    df = df[df.index >= START_DATE]

    # Features: ffill + bfill puis drop colonnes encore NA
    df_features = df.ffill().bfill()
    df_features = df_features.dropna(axis=1)

    # Target: rendements futurs BTC
    if "XBTUSD Curncy" not in df.columns:
        raise ValueError("Colonne BTC introuvable: 'XBTUSD Curncy'")

    btc_price = df["XBTUSD Curncy"]
    returns = calculate_returns(btc_price, horizon, return_type)

    # Target classification: scénarios
    scenarios = classify_scenario(returns.dropna())
    df_features["target_class"] = scenarios

    # Nettoyage final
    df_clean = df_features.dropna(subset=["target_class"])
    if len(df_clean) < 100:
        return None

    y_class = df_clean["target_class"].astype(int)

    X = df_clean.drop(columns=["target_class"])

    # Retirer colonnes potentiellement problématiques / leakage-like
    X = X.drop(columns=[c for c in X.columns if "logret" in c.lower() or "logreturn" in c.lower()], errors="ignore")

    # Retirer colonnes BTC price (si elles existent)
    btc_cols = [c for c in X.columns if "XBTUSD" in c]
    X = X.drop(columns=btc_cols, errors="ignore")

    # Split temporel
    train_size = int(0.8 * len(X))
    X_train = X.iloc[:train_size]
    X_test = X.iloc[train_size:]
    y_train = y_class.iloc[:train_size]
    y_test = y_class.iloc[train_size:]

    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Modèle classification
    rf_class = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    rf_class.fit(X_train_scaled, y_train)
    y_pred = rf_class.predict(X_test_scaled)

    # Métriques
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    # Importance features
    imp_class = pd.DataFrame({
        "feature": X_train.columns,
        "importance": rf_class.feature_importances_
    }).sort_values("importance", ascending=False)

    # Distributions
    scenario_dist_train = y_train.value_counts().sort_index()
    scenario_dist_test = y_test.value_counts().sort_index()

    # Stats returns (utile pour commentaire)
    ret_stats = {
        "mean": float(returns.mean()),
        "std": float(returns.std()),
        "min": float(returns.min()),
        "max": float(returns.max()),
        "skew": float(returns.skew()),
        "kurt": float(returns.kurtosis())
    }

    return {
        "horizon": horizon,
        "n_samples": len(df_clean),
        "n_test": len(y_test),

        "acc_class": acc,
        "confusion_matrix": cm,
        "top_features_class": imp_class.head(10),

        "scenario_dist_train": scenario_dist_train,
        "scenario_dist_test": scenario_dist_test,

        "return_stats": ret_stats,

        "y_class_test": y_test,
        "y_class_pred": y_pred
    }

# ============================================================
# EXÉCUTION MULTI-HORIZONS (CLASSIFICATION UNIQUEMENT)
# ============================================================

print("=" * 80)
print(" ANALYSE MULTI-HORIZONS - CLASSIFICATION SCÉNARIOS UNIQUEMENT")
print("=" * 80)

all_results = []
all_features_class = defaultdict(list)  # Tracker les features importantes en classification

for h in HORIZONS:
    print(f"\n{'=' * 80}")
    print(f" Traitement horizon {h} jours...")
    print(f"{'=' * 80}")

    res = evaluate_horizon_classification_only(pivot_df, h, return_type=RETURN_TYPE)

    if res is not None:
        all_results.append(res)

        # Tracker top features (top 5)
        for _, row in res["top_features_class"].head(5).iterrows():
            all_features_class[row["feature"]].append((h, row["importance"]))

        print(f"\n CLASSIFICATION (scénarios):")
        print(f"   Accuracy = {res['acc_class'] * 100:.1f}%")

        print(f"\n Distribution des scénarios TEST:")
        scenario_names = ["Fort Baissier", "Baissier", "Neutre", "Haussier", "Fort Haussier"]
        total_test = len(res["y_class_test"])
        for idx, count in res["scenario_dist_test"].items():
            print(f"   {scenario_names[int(idx)]}: {count} ({count / total_test * 100:.1f}%)")

        print(f"\n Top 5 features (CLASSIFICATION):")
        print(res["top_features_class"].head(5).to_string(index=False))

    else:
        print(f"    Pas assez de données pour horizon {h}")

print("\n" + "=" * 80)
print(" ANALYSE TERMINÉE")
print("=" * 80)

# ============================================================
# TABLEAU RÉCAPITULATIF (CLASSIFICATION UNIQUEMENT)
# ============================================================

summary_df = pd.DataFrame([
    {
        "Horizon (j)": r["horizon"],
        "N échantillons": r["n_samples"],
        "Acc Scénarios (%)": r["acc_class"] * 100,
        "Mean Return": r["return_stats"]["mean"],
        "Std Return": r["return_stats"]["std"]
    }
    for r in all_results
])

print("\n" + "=" * 80)
print(" TABLEAU RÉCAPITULATIF")
print("=" * 80)
print(summary_df.to_string(index=False))

# ============================================================
# FEATURES LES PLUS RÉCURRENTES (CLASSIFICATION)
# ============================================================

print("\n" + "=" * 80)
print(" FEATURES LES PLUS RÉCURRENTES (top 5 sur plusieurs horizons)")
print("=" * 80)

feature_counts_class = {feat: len(horizons) for feat, horizons in all_features_class.items()}
sorted_features_class = sorted(feature_counts_class.items(), key=lambda x: x[1], reverse=True)

for feat, count in sorted_features_class[:15]:
    horizons_str = ", ".join([f"{h}j" for h, _ in all_features_class[feat]])
    avg_imp = float(np.mean([imp for _, imp in all_features_class[feat]]))
    print(f"  {feat:30} → Présent dans {count}/{len(HORIZONS)} horizons | Imp moy: {avg_imp:.4f}")
    print(f"     Horizons: {horizons_str}")

print("\n" + "=" * 80)


 ANALYSE MULTI-HORIZONS - CLASSIFICATION SCÉNARIOS UNIQUEMENT

 Traitement horizon 1 jours...

 CLASSIFICATION (scénarios):
   Accuracy = 50.1%

 Distribution des scénarios TEST:
   Fort Baissier: 31 (5.9%)
   Baissier: 80 (15.3%)
   Neutre: 294 (56.2%)
   Haussier: 85 (16.3%)
   Fort Haussier: 33 (6.3%)

 Top 5 features (CLASSIFICATION):
        feature  importance
 RIOT US Equity    0.014985
CO1 COMB Comdty    0.014134
     SXXP Index    0.013908
NG1 COMB Comdty    0.013786
       TxTfrCnt    0.013502

 Traitement horizon 14 jours...

 CLASSIFICATION (scénarios):
   Accuracy = 52.9%

 Distribution des scénarios TEST:
   Fort Baissier: 28 (5.2%)
   Baissier: 93 (17.3%)
   Neutre: 305 (56.6%)
   Haussier: 86 (16.0%)
   Fort Haussier: 27 (5.0%)

 Top 5 features (CLASSIFICATION):
             feature  importance
       XBNUSD Curncy    0.023067
       XLCUSD Curncy    0.018927
days_to_next_halving    0.015975
  days_since_halving    0.015601
       XRPUSD Curncy    0.015229

 Traitement 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# VISUALISATIONS COMPLÈTES
# ============================================================

# Supposons que all_results est disponible de l'exécution précédente
# Sinon, charger depuis un pickle

print("="*80)
print(" GÉNÉRATION DES VISUALISATIONS")
print("="*80)

# Style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 12)

# ============================================================
# FIGURE 1: PERFORMANCE EN FONCTION DE L'HORIZON
# ============================================================

fig1, axes = plt.subplots(2, 3, figsize=(18, 10))
fig1.suptitle(" Performance des Modèles en Fonction de l'Horizon Temporel", 
              fontsize=16, fontweight='bold')

horizons = [r["horizon"] for r in all_results]
r2_values = [r["r2"] for r in all_results]
rmse_values = [r["rmse"] for r in all_results]
dir_acc_reg = [r["dir_acc_reg"]*100 for r in all_results]
acc_class = [r["acc_class"]*100 for r in all_results]
mean_returns = [r["return_stats"]["mean"]*100 for r in all_results]
std_returns = [r["return_stats"]["std"]*100 for r in all_results]

# 1. R²
axes[0, 0].plot(horizons, r2_values, 'o-', linewidth=2, markersize=10, color='#2E86AB')
axes[0, 0].axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Baseline')
axes[0, 0].set_xlabel('Horizon (jours)', fontsize=11)
axes[0, 0].set_ylabel('R²', fontsize=11)
axes[0, 0].set_title('R² (Régression)', fontweight='bold')
axes[0, 0].grid(alpha=0.3)
axes[0, 0].legend()

# 2. RMSE
axes[0, 1].plot(horizons, rmse_values, 'o-', linewidth=2, markersize=10, color='#A23B72')
axes[0, 1].set_xlabel('Horizon (jours)', fontsize=11)
axes[0, 1].set_ylabel('RMSE', fontsize=11)
axes[0, 1].set_title('RMSE (Régression)', fontweight='bold')
axes[0, 1].grid(alpha=0.3)

# 3. Directional Accuracy (Régression)
axes[0, 2].plot(horizons, dir_acc_reg, 'o-', linewidth=2, markersize=10, color='#F18F01')
axes[0, 2].axhline(y=50, color='red', linestyle='--', alpha=0.5, label='Hasard (50%)')
axes[0, 2].set_xlabel('Horizon (jours)', fontsize=11)
axes[0, 2].set_ylabel('Accuracy (%)', fontsize=11)
axes[0, 2].set_title('Directional Accuracy (Régression)', fontweight='bold')
axes[0, 2].grid(alpha=0.3)
axes[0, 2].legend()
axes[0, 2].set_ylim(30, 75)

# 4. Accuracy Classification (Scénarios)
axes[1, 0].plot(horizons, acc_class, 'o-', linewidth=2, markersize=10, color='#6A994E')
axes[1, 0].axhline(y=20, color='red', linestyle='--', alpha=0.5, label='Hasard (20%)')
axes[1, 0].set_xlabel('Horizon (jours)', fontsize=11)
axes[1, 0].set_ylabel('Accuracy (%)', fontsize=11)
axes[1, 0].set_title('Accuracy Classification (5 scénarios)', fontweight='bold')
axes[1, 0].grid(alpha=0.3)
axes[1, 0].legend()

# 5. Mean Return
axes[1, 1].plot(horizons, mean_returns, 'o-', linewidth=2, markersize=10, color='#BC4B51')
axes[1, 1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[1, 1].set_xlabel('Horizon (jours)', fontsize=11)
axes[1, 1].set_ylabel('Rendement moyen (%)', fontsize=11)
axes[1, 1].set_title('Rendement Moyen par Horizon', fontweight='bold')
axes[1, 1].grid(alpha=0.3)

# 6. Std Return (Volatilité)
axes[1, 2].plot(horizons, std_returns, 'o-', linewidth=2, markersize=10, color='#8B5A3C')
axes[1, 2].set_xlabel('Horizon (jours)', fontsize=11)
axes[1, 2].set_ylabel('Volatilité (%)', fontsize=11)
axes[1, 2].set_title('Volatilité des Rendements par Horizon', fontweight='bold')
axes[1, 2].grid(alpha=0.3)

plt.tight_layout()

# ============================================================
# FIGURE 2: MATRICES DE CONFUSION (1 SEMAINE)
# ============================================================

# Trouver les résultats pour horizon = 7 jours
res_7d = next((r for r in all_results if r["horizon"] == 7), None)

if res_7d is not None:
    fig2, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    cm = res_7d["confusion_matrix"]
    scenario_names = ["Fort\nBaissier", "Baissier", "Neutre", "Haussier", "Fort\nHaussier"]
    
    # Normaliser par ligne (pourcentage par classe réelle)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    sns.heatmap(cm_normalized, annot=True, fmt='.1f', cmap='YlOrRd', 
                xticklabels=scenario_names, yticklabels=scenario_names,
                cbar_kws={'label': 'Pourcentage (%)'}, ax=ax)
    
    ax.set_xlabel('Scénario Prédit', fontsize=12, fontweight='bold')
    ax.set_ylabel('Scénario Réel', fontsize=12, fontweight='bold')
    ax.set_title('Matrice de Confusion - Horizon 1 Semaine (7 jours)\n(Pourcentages par ligne)', 
                 fontsize=14, fontweight='bold')
    
    plt.tight_layout()

# ============================================================
# FIGURE 3: DISTRIBUTION DES SCÉNARIOS PAR HORIZON
# ============================================================

fig3, axes = plt.subplots(2, 4, figsize=(18, 10))
fig3.suptitle(" Distribution des Scénarios par Horizon Temporel", 
              fontsize=16, fontweight='bold')

axes = axes.flatten()
scenario_names_short = ["Fort Baiss.", "Baissier", "Neutre", "Haussier", "Fort Hauss."]
colors = ['#d62828', '#f77f00', '#fcbf49', '#06d6a0', '#118ab2']

for idx, res in enumerate(all_results):
    if idx < len(axes):
        dist = res["scenario_dist_test"]
        
        # S'assurer que tous les scénarios sont présents
        full_dist = pd.Series([0]*5, index=range(5))
        full_dist.update(dist)
        
        axes[idx].bar(range(5), full_dist.values, color=colors, alpha=0.8, edgecolor='black')
        axes[idx].set_xlabel('Scénario', fontsize=10)
        axes[idx].set_ylabel('Nombre', fontsize=10)
        axes[idx].set_title(f'Horizon {res["horizon"]} jours', fontweight='bold')
        axes[idx].set_xticks(range(5))
        axes[idx].set_xticklabels(scenario_names_short, rotation=45, ha='right', fontsize=8)
        axes[idx].grid(alpha=0.3, axis='y')

# Cacher les axes vides
for idx in range(len(all_results), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()

# ============================================================
# FIGURE 4: ÉVOLUTION DES TOP FEATURES
# ============================================================

# Identifier les 10 features les plus récurrentes
top_recurring_features_reg = sorted_features_reg[:10]

fig4, ax = plt.subplots(1, 1, figsize=(14, 8))

# Pour chaque feature récurrente, tracer son importance en fonction de l'horizon
for feat, _ in top_recurring_features_reg:
    if feat in all_features_reg:
        horizons_feat = [h for h, _ in all_features_reg[feat]]
        importances_feat = [imp for _, imp in all_features_reg[feat]]
        
        ax.plot(horizons_feat, importances_feat, 'o-', label=feat, linewidth=2, markersize=8)

ax.set_xlabel('Horizon (jours)', fontsize=12, fontweight='bold')
ax.set_ylabel('Importance', fontsize=12, fontweight='bold')
ax.set_title('Évolution de l\'Importance des Top 10 Features (Régression)', 
             fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()

# ============================================================
# FIGURE 5: PRÉDICTIONS VS RÉALITÉ (HORIZON 7j et 30j)
# ============================================================

fig5, axes = plt.subplots(1, 2, figsize=(16, 6))

# Horizon 7 jours
res_7d = next((r for r in all_results if r["horizon"] == 7), None)
if res_7d:
    y_true = res_7d["y_reg_test"].values * 100  # Convertir en %
    y_pred = res_7d["y_reg_pred"] * 100
    
    axes[0].scatter(y_true, y_pred, alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
    
    # Ligne de prédiction parfaite
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Prédiction parfaite')
    
    axes[0].set_xlabel('Rendement Réel (%)', fontsize=11, fontweight='bold')
    axes[0].set_ylabel('Rendement Prédit (%)', fontsize=11, fontweight='bold')
    axes[0].set_title(f'Horizon 7 jours (R²={res_7d["r2"]:.3f})', fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

# Horizon 30 jours
res_30d = next((r for r in all_results if r["horizon"] == 30), None)
if res_30d:
    y_true = res_30d["y_reg_test"].values * 100
    y_pred = res_30d["y_reg_pred"] * 100
    
    axes[1].scatter(y_true, y_pred, alpha=0.6, s=50, edgecolors='black', linewidth=0.5, color='orange')
    
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Prédiction parfaite')
    
    axes[1].set_xlabel('Rendement Réel (%)', fontsize=11, fontweight='bold')
    axes[1].set_ylabel('Rendement Prédit (%)', fontsize=11, fontweight='bold')
    axes[1].set_title(f'Horizon 30 jours (R²={res_30d["r2"]:.3f})', fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

plt.tight_layout()

print("\n" + "="*80)
print(" TOUTES LES VISUALISATIONS GÉNÉRÉES")
print("="*80)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Choisis l'horizon que tu veux voir
horizon_choisi = 30  # Change ça pour 14, 30, 60, 90, 120, 150

# Trouve les résultats pour cet horizon
res = next((r for r in all_results if r["horizon"] == horizon_choisi), None)

if res is not None:
    # Récupère la matrice de confusion
    cm = res["confusion_matrix"]
    
    # Noms des scénarios
    scenario_names = ["Fort\nBaissier", "Baissier", "Neutre", "Haussier", "Fort\nHaussier"]
    
    # Normalise par ligne (pourcentage par classe réelle)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    # Crée la figure
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_normalized, 
                annot=True,           # Affiche les valeurs
                fmt='.1f',            # 1 décimale
                cmap='YlOrRd',        # Palette de couleurs
                xticklabels=scenario_names, 
                yticklabels=scenario_names,
                cbar_kws={'label': 'Pourcentage (%)'},
                vmin=0, vmax=100)
    
    plt.xlabel('Scénario Prédit', fontsize=12, fontweight='bold')
    plt.ylabel('Scénario Réel', fontsize=12, fontweight='bold')
    plt.title(f'Matrice de Confusion - Horizon {horizon_choisi} jours\n(Pourcentages par ligne)', 
              fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Affiche aussi les valeurs brutes
    print(f"\n Matrice Brute (Nombre d'échantillons):")
    print(cm)
    
    # Affiche l'accuracy
    accuracy = np.trace(cm) / np.sum(cm) * 100
    print(f"\n Accuracy: {accuracy:.1f}%")
else:
    print(f" Pas de résultats pour horizon {horizon_choisi}j")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Choisis ton horizon
horizon_choisi = 14  # Change pour 14, 30, 60, 90, 120, 150

# Trouve les résultats
res = next((r for r in all_results if r["horizon"] == horizon_choisi), None)

if res is not None:
    # Récupère la matrice de confusion de la CLASSIFICATION
    cm = res["confusion_matrix"]
    
    # CORRECTION : Adapte les noms selon la taille réelle de la matrice
    scenario_names_full = ["Fort Baissier", "Baissier", "Neutre", "Haussier", "Fort Haussier"]
    
    # Nombre de scénarios réellement présents
    n_scenarios = cm.shape[0]
    
    # Prend seulement les noms correspondants
    scenario_names = [scenario_names_full[i] for i in range(n_scenarios)]
    scenario_names_plot = [s.replace(" ", "\n") for s in scenario_names]
    
    # Normalise par ligne (% par classe réelle)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    # Crée la figure
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_normalized, 
                annot=True,
                fmt='.1f',
                cmap='RdYlGn',
                xticklabels=scenario_names_plot, 
                yticklabels=scenario_names_plot,
                cbar_kws={'label': 'Pourcentage (%)'},
                vmin=0, vmax=100)
    
    plt.xlabel('Scénario Prédit (Classification)', fontsize=12, fontweight='bold')
    plt.ylabel('Scénario Réel', fontsize=12, fontweight='bold')
    plt.title(f'Matrice de Confusion - Classification\nHorizon {horizon_choisi} jours ({n_scenarios} scénarios présents)', 
              fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Stats
    accuracy = np.trace(cm) / np.sum(cm) * 100
    print(f"\n Accuracy Classification: {accuracy:.1f}%")
    print(f" Nombre total d'échantillons: {np.sum(cm)}")
    print(f" Nombre de scénarios présents: {n_scenarios}/5")
    
    # Distribution des prédictions
    print(f"\n Distribution des PRÉDICTIONS:")
    pred_dist = cm.sum(axis=0)
    for i in range(len(pred_dist)):
        pct = (pred_dist[i] / pred_dist.sum()) * 100
        print(f"   {scenario_names[i]}: {pred_dist[i]} ({pct:.1f}%)")
    
    # Distribution des réalités
    print(f"\n Distribution des RÉALITÉS:")
    real_dist = cm.sum(axis=1)
    for i in range(len(real_dist)):
        pct = (real_dist[i] / real_dist.sum()) * 100
        print(f"   {scenario_names[i]}: {real_dist[i]} ({pct:.1f}%)")
    
    # Scénarios manquants
    if n_scenarios < 5:
        missing = [scenario_names_full[i] for i in range(5) if i >= n_scenarios or i not in range(n_scenarios)]
        print(f"\n Scénarios ABSENTS du test set:")
        for m in missing:
            print(f"   • {m}")
else:
    print(f" Pas de résultats pour horizon {horizon_choisi}j")

## test causality

In [ ]:
pip install graphviz 

In [ ]:
# ===============================
# CAUSAL PIPELINE (DoWhy) - VERSION CORRIGÉE v2
# ===============================

import pandas as pd
import numpy as np

# 1) Base DF
df = pivot_df.copy()

# Ensure datetime index
df.index = pd.to_datetime(df.index)

# 2) Create BTC log-return + horizon target (choose one)
HORIZON = 7  # change to 1, 7, 30, etc.

if "XBTUSD Curncy" not in df.columns:
    raise ValueError("BTC price column 'XBTUSD Curncy' missing.")

df["BTC_logret"] = np.log(df["XBTUSD Curncy"]).diff()

# Target = cumulative log-return over next HORIZON days
df["target"] = df["BTC_logret"].rolling(HORIZON).sum().shift(-HORIZON)

# Drop only where target or BTC is missing
df = df.dropna(subset=["BTC_logret", "target"])

# 3) OPTIONAL: filter date (if you want after 2018)
# df = df[df.index >= "2018-03-01"]

# 4) Fill missing values in features (important for macro series)
df = df.ffill().bfill()

# Remove columns you don't want as features
drop_cols = ["BTC_logret", "XBTUSD Curncy"]  # keep target

# -------------------------------
# Choose treatment candidates
# -------------------------------
treatments = [
    "VIX Index",
    "MOVE Index",
    "DXY Curncy",
    "USYC2Y10 Index",
    "DEYC2Y10 Index"
]

# keep only treatments present
treatments = [t for t in treatments if t in df.columns]
print("Treatments available:", treatments)

# -------------------------------
# Choose confounders (controls)
# -------------------------------
confounders = []
candidate_confounders = [
    "SPX Index", "CCMP Index", "RTY Index",
    "CL1 COMB Comdty", "NG1 COMB Comdty",
    "EURUSD Curncy", "JPYUSD Curncy", "CNH Curncy",
    "M2 Index", "ECMAM2 Index",
    "WGDPUS Index", "WGDPEURO Index"
]

confounders = [c for c in candidate_confounders if c in df.columns]
print("Confounders used:", confounders)

# -------------------------------
# Make causal dataset
# -------------------------------
causal_df = df[["target"] + treatments + confounders].copy()

# Ensure numeric
for col in causal_df.columns:
    causal_df[col] = pd.to_numeric(causal_df[col], errors="coerce")

causal_df = causal_df.dropna()
print("Causal df shape:", causal_df.shape)

# ===============================
# DoWhy Causal Estimation - SOLUTION GML FORMAT
# ===============================
from dowhy import CausalModel

results = []

for T in treatments:
    print("\n==============================")
    print("Treatment:", T, "| Outcome: target | Horizon:", HORIZON)

    # --------------------------
    # Créer le graphe en format GML (celui qui marche!)
    # --------------------------
    gml_graph = "graph [\n  directed 1\n"
    
    # Ajouter les nœuds
    gml_graph += f'  node [ id "{T}" label "{T}" ]\n'
    gml_graph += f'  node [ id "target" label "target" ]\n'
    
    for c in confounders:
        gml_graph += f'  node [ id "{c}" label "{c}" ]\n'
    
    # Ajouter les arêtes
    for c in confounders:
        gml_graph += f'  edge [ source "{c}" target "{T}" ]\n'
        gml_graph += f'  edge [ source "{c}" target "target" ]\n'
    
    gml_graph += f'  edge [ source "{T}" target "target" ]\n'
    gml_graph += "]\n"

    # Créer le modèle avec le graphe GML
    model = CausalModel(
        data=causal_df,
        treatment=T,
        outcome="target",
        graph=gml_graph
    )

    identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
    print("Identified estimand:", identified_estimand)

    # --------------------------
    # Estimate effect (backdoor)
    # --------------------------
    estimate = model.estimate_effect(
        identified_estimand,
        method_name="backdoor.linear_regression"
    )

    print("Causal estimate (ATE):", estimate.value)

    # --------------------------
    # Refutation tests
    # --------------------------
    try:
        ref_placebo = model.refute_estimate(
            identified_estimand, estimate,
            method_name="placebo_treatment_refuter",
            placebo_type="permute"
        )
        placebo_new_est = getattr(ref_placebo, "new_effect", None)
        placebo_p = None
        if hasattr(ref_placebo, 'refutation_result'):
            placebo_p = ref_placebo.refutation_result.get('p_value', None)
        print("\nRefutation — placebo:", ref_placebo)
    except Exception as e:
        print(f"Placebo test failed: {e}")
        placebo_new_est = None
        placebo_p = None

    try:
        ref_randomcc = model.refute_estimate(
            identified_estimand, estimate,
            method_name="random_common_cause"
        )
        randomcc_new_est = getattr(ref_randomcc, "new_effect", None)
        print("Refutation — random common cause:", ref_randomcc)
    except Exception as e:
        print(f"Random common cause test failed: {e}")
        randomcc_new_est = None

    try:
        ref_subset = model.refute_estimate(
            identified_estimand, estimate,
            method_name="data_subset_refuter",
            subset_fraction=0.8
        )
        subset_new_est = getattr(ref_subset, "new_effect", None)
        print("Refutation — data subset:", ref_subset)
    except Exception as e:
        print(f"Subset test failed: {e}")
        subset_new_est = None

    results.append({
        "Horizon": HORIZON,
        "Treatment": T,
        "ATE": estimate.value,
        "Placebo_new_est": placebo_new_est,
        "Placebo_p": placebo_p,
        "RandomCC_new_est": randomcc_new_est,
        "Subset_new_est": subset_new_est
    })

# ===============================
# Results table
# ===============================
results_df = pd.DataFrame(results).sort_values("ATE", ascending=False, key=abs)
print("\n=== SUMMARY RESULTS ===")
display(results_df)

# Optional: save
# results_df.to_csv(f"causal_results_h{HORIZON}.csv", index=False)
# print(f"Saved: causal_results_h{HORIZON}.csv")

In [ ]:
# ===============================
# CAUSAL PIPELINE
# ===============================

import pandas as pd
import numpy as np
from dowhy import CausalModel

# 1) Base DF
df = pivot_df.copy()
df.index = pd.to_datetime(df.index)

# =================================
HORIZON = 30  # 1, 7, 30, 90, etc.
# =================================

if "XBTUSD Curncy" not in df.columns:
    raise ValueError("BTC price column 'XBTUSD Curncy' missing.")

df["BTC_logret"] = np.log(df["XBTUSD Curncy"]).diff()
df["target"] = df["BTC_logret"].rolling(HORIZON).sum().shift(-HORIZON)
df = df.dropna(subset=["BTC_logret", "target"])
df = df.ffill().bfill()

# Treatments
treatments = [
    "VIX Index",
    "MOVE Index", 
    "DXY Curncy",
    "USYC2Y10 Index",
    "DEYC2Y10 Index"
]
treatments = [t for t in treatments if t in df.columns]

# Confounders
candidate_confounders = [
    "SPX Index", "CCMP Index", "RTY Index",
    "CL1 COMB Comdty", "NG1 COMB Comdty",
    "EURUSD Curncy", "JPYUSD Curncy", "CNH Curncy",
    "M2 Index", "ECMAM2 Index",
    "WGDPUS Index", "WGDPEURO Index"
]
confounders = [c for c in candidate_confounders if c in df.columns]

# Causal dataset
causal_df = df[["target"] + treatments + confounders].copy()
for col in causal_df.columns:
    causal_df[col] = pd.to_numeric(causal_df[col], errors="coerce")
causal_df = causal_df.dropna()

print(f"\n{'='*50}")
print(f"HORIZON: {HORIZON} days | Sample size: {len(causal_df)}")
print(f"{'='*50}\n")

# ===============================
# LOOP - AFFICHAGE MINIMAL
# ===============================

results = []

for T in treatments:
    # Build GML graph
    gml_graph = "graph [\n  directed 1\n"
    gml_graph += f'  node [ id "{T}" label "{T}" ]\n'
    gml_graph += f'  node [ id "target" label "target" ]\n'
    
    for c in confounders:
        gml_graph += f'  node [ id "{c}" label "{c}" ]\n'
    
    for c in confounders:
        gml_graph += f'  edge [ source "{c}" target "{T}" ]\n'
        gml_graph += f'  edge [ source "{c}" target "target" ]\n'
    
    gml_graph += f'  edge [ source "{T}" target "target" ]\n'
    gml_graph += "]\n"

    # Causal model
    model = CausalModel(
        data=causal_df,
        treatment=T,
        outcome="target",
        graph=gml_graph
    )

    identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
    estimate = model.estimate_effect(
        identified_estimand,
        method_name="backdoor.linear_regression"
    )

    # Refutation tests (silent)
    try:
        ref_placebo = model.refute_estimate(
            identified_estimand, estimate,
            method_name="placebo_treatment_refuter",
            placebo_type="permute"
        )
        placebo_new_est = getattr(ref_placebo, "new_effect", None)
        placebo_p = ref_placebo.refutation_result.get('p_value', None) if hasattr(ref_placebo, 'refutation_result') else None
    except:
        placebo_new_est = None
        placebo_p = None

    try:
        ref_randomcc = model.refute_estimate(
            identified_estimand, estimate,
            method_name="random_common_cause"
        )
        randomcc_new_est = getattr(ref_randomcc, "new_effect", None)
    except:
        randomcc_new_est = None

    try:
        ref_subset = model.refute_estimate(
            identified_estimand, estimate,
            method_name="data_subset_refuter",
            subset_fraction=0.8
        )
        subset_new_est = getattr(ref_subset, "new_effect", None)
    except:
        subset_new_est = None

    results.append({
        "Horizon": HORIZON,
        "Treatment": T,
        "ATE": estimate.value,
        "Placebo_new_est": placebo_new_est,
        "Placebo_p": placebo_p,
        "RandomCC_new_est": randomcc_new_est,
        "Subset_new_est": subset_new_est
    })
    
    # Affichage minimal par traitement
    print(f"{T:20s} | ATE: {estimate.value:10.6f}")

# ===============================
# RESULTS
# ===============================
results_df = pd.DataFrame(results).sort_values("ATE", ascending=False, key=abs)

print(f"\n{'='*50}")
print("=== SUMMARY RESULTS ===")
print(f"{'='*50}")
display(results_df)

# Save
results_df.to_csv(f"causal_results_h{HORIZON}.csv", index=False)
print(f"\n✓ Saved: causal_results_h{HORIZON}.csv")